In [3]:
DATASETS = [
    {'path': '/media/alvarinho/dados/Datasets/raw/emea en-es.txt/', 'src_file': 'EMEA.en-es.en', 'tgt_file': 'EMEA.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/emea en-pt.txt/', 'src_file': 'EMEA.en-pt.en', 'tgt_file': 'EMEA.en-pt.pt'},
 
    {'path': '/media/alvarinho/dados/Datasets/raw/paracrawl en-es.txt/', 'src_file': 'ParaCrawl.en-es.en', 'tgt_file': 'ParaCrawl.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/paracrawl en-pt.txt/', 'src_file': 'ParaCrawl.en-pt.en', 'tgt_file': 'ParaCrawl.en-pt.pt'},
    
    {'path': '/media/alvarinho/dados/Datasets/raw/scielo en-pt.txt/', 'src_file': 'SciELO.en-pt.en', 'tgt_file': 'SciELO.en-pt.pt'},

    {'path': '/media/alvarinho/dados/Datasets/raw/wikimatrix en-es.txt/', 'src_file': 'WikiMatrix.en-es.en', 'tgt_file': 'WikiMatrix.en-es.es'},
    {'path': '/media/alvarinho/dados/Datasets/raw/wikimatrix en-pt.txt/', 'src_file': 'WikiMatrix.en-pt.en', 'tgt_file': 'WikiMatrix.en-pt.pt'},
]

In [4]:
from src.clean_character import CleanCharacter
from src.deduplicator import ExactDuplicator, MinHashDetector
from src.lang_identify import LangIdentifier
from src.length_clash import LengthClash
from src.numbers_clash import NumbersClash
from src.unidecode_norm import UnidecodeNorm
from typing import List
from src.processor import Processor
import pandas as pd

In [ ]:
class Pipeline:
    def __init__(self):
        self.steps: List[Processor] = []

    def add_step(self, step):
        self.steps.append(step)

    def process_all(self, kwargs):
        result_total = []
        metrics = {}
        for step in self.steps:
            step_name = step.__class__.__name__
            # print('Processando step:', )
            # print('Textos:', kwargs['text1'], kwargs['text2'])
            (text1, text2), eval, step_metrics = step.apply_pairs(**kwargs)
            result_total.append(eval)
            metrics.update(step_metrics)
            kwargs['text1'] = text1
            kwargs['text2'] = text2
            
            # print(f"   EVAL: {eval}")
        return kwargs, all(result_total), metrics
    def process(self, kwargs):
        result_total = []
        for step in self.steps:
            # print('Processando step:', )
            (text1, text2), eval, _ = step.apply_pairs(**kwargs)
            result_total.append(eval)
            if(not eval):
                break

            kwargs['text1'] = text1
            kwargs['text2'] = text2
            
            # print(f"   EVAL: {eval}")
        return kwargs, all(result_total)

pipe = Pipeline()
pipe.add_step(CleanCharacter())
pipe.add_step(UnidecodeNorm())
pipe.add_step(LangIdentifier(model_path=None, threshold=0.75, k=1))
pipe.add_step(LengthClash(max_length_diff_ratio=2.0))
# pipe.add_step(ExactDuplicator())
# pipe.add_step(MinHashDetector())
pipe.add_step(NumbersClash(threshold=0.75))

In [8]:
dataset_amostra = DATASETS[0]

In [9]:
path = dataset_amostra['path']
src_file = dataset_amostra['src_file']
tgt_file = dataset_amostra['tgt_file']
df_analise = pd.DataFrame()

with open(path + src_file, 'r', encoding='utf-8') as f_src, open(path + tgt_file, 'r', encoding='utf-8') as f_tgt:
    for i, (src_line, tgt_line) in enumerate(zip(f_src, f_tgt)):
        _, eval, metrics = pipe.process_all(
            {'text1': src_line.strip(), 
             'text2': tgt_line.strip(), 
             'expected_lang1': 'en', 
             'expected_lang2': 'es'
             })  

        df_analise_atual = pd.DataFrame({
            'path': [path],
            'src_file': [src_file],
            'tgt_file': [tgt_file],
            'eval': [eval]})
        df_analise_atual_metrics = pd.DataFrame.from_dict(metrics, orient='index')
        df_analise_atual = pd.concat([df_analise_atual, df_analise_atual_metrics], axis=0)
        df_analise = pd.concat([df_analise, df_analise_atual], ignore_index=True)

        print(f"{i+1}: SRC: {src_line.strip()}")
        print(f"    TGT: {tgt_line.strip()}")
        print(f"    EVAL: {eval}")  
        print(f"    METRICS: {metrics}")
        print("--------------------------------------------------")
        print("--------------------------------------------------")

        if i >= 500:  # Limitar a 10 linhas para teste
            break


Textos: European Medicines Agency European Medicines Agency
Textos: European Medicines Agency European Medicines Agency
Textos: European Medicines Agency European Medicines Agency
Textos: European Medicines Agency European Medicines Agency
Textos: European Medicines Agency European Medicines Agency
1: SRC: European Medicines Agency
    TGT: European Medicines Agency
    EVAL: False
    METRICS: {'lang_identify_prob1': 0, 'lang_identify_prob2': 0, 'length_clash_diff_ratio': 1.0, 'number_clash_similarity': 0}
--------------------------------------------------
--------------------------------------------------
Textos: EMEA/ H/ C/ 471 EMEA/ H/ C/ 471
Textos: EMEA/ H/ C/ 471 EMEA/ H/ C/ 471
Textos: EMEA/ H/ C/ 471 EMEA/ H/ C/ 471
Textos: EMEA/ H/ C/ 471 EMEA/ H/ C/ 471
Textos: EMEA/ H/ C/ 471 EMEA/ H/ C/ 471
2: SRC: EMEA/ H/ C/ 471
    TGT: EMEA/ H/ C/ 471
    EVAL: False
    METRICS: {'lang_identify_prob1': 0, 'lang_identify_prob2': 0, 'length_clash_diff_ratio': 1.0, 'number_clash_similari

In [6]:
df_analise_atual_metrics

,0
lang_identify_prob1,0.0
lang_identify_prob2,0.0
length_clash_diff_ratio,1.4
number_clash_similarity,1.0
